In [ ]:
import pandas as pd

import config as c

data_local = "C:/Users/glori/Documents/Persönliches/#PhD_local/code/reddit_bd_recovery/data/"

In [ ]:
# build PR-BD corpus

# 1) posts in BD subreddits
# subreddit_type based on categorisation here: https://github.com/glorisonne/reddit_bd_mood_posting_mh/blob/main/data/subreddit_topics.csv
# (Fourth level = "bipolar")
#posts = pd.read_csv(c.data + "posts_meta.csv", usecols=["id", "user_id", "subreddit_name", "text_wordcount", "mentions_bd",
#                                                        "subreddit_type"])

#posts = posts[posts.subreddit_type == "bd"]
#print("Posts in BD subreddits:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))

# 2) posts that mention BD -- refer to BD terms list https://github.com/glorisonne/reddit_bd_user_characteristics/blob/master/disclosure-patterns/condition-terms/bipolar-filter-terms.txt
#posts = posts[posts.mentions_bd]
#print("Posts that mention BD:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))

# 3) posts with at least 94 words, duplicates (same text by same user) removed
#posts = posts[posts.text_wordcount > 93]

# add post texts - read post_text csv in two batches as the file is very large and the code migth therefore fail on
# machines with less RAM
#posts_text_0 = pd.read_csv(c.data + "posts_text.csv", nrows=10000000)
#posts_text_0 = posts_text_0[posts_text_0.id.isin(posts.id)]
#posts_text_1 = pd.read_csv(c.data + "posts_text.csv", skiprows=10000000, header=None, names=["id", "text"])
#posts_text_1 = posts_text_1[posts_text_1.id.isin(posts.id)]
#posts_text = pd.concat([posts_text_0, posts_text_1])

#posts = posts.merge(posts_text, left_on="id", right_on="id")

#del(posts_text_0, posts_text_1, posts_text)

#posts.drop_duplicates(subset=["text", "user_id"], keep = "last", inplace=True)
#print("Posts with at least 94 words, duplicates removed:\nPosts: %d\nWords: %d\nUsers: %d" %(len(posts), posts.text_wordcount.sum(),
#                                                                    posts.user_id.nunique()))
posts.to_csv(data_local + "posts_bd.csv")

# TBD lemmatize (check when did this for the wordcount in the flowchart)
# preprocess_posts_with_spacy.py. nohup python process_posts_with_spacy.py data/posts_bd.csv &> output/process_posts_spacy.log &
comBine output files
PR_scoring.py

# 4) cosine similarity with PR terms list - share PR terms list in data/ (ToDo: possibly share "standard" list with variants + list of only unique terms)

# 5) Reference corpus - cosine similarity cutoff

In [ ]:
# figure out where different number of posts comes from for BD subreddit posts

# lower/uppercasing does not matter for the BD subreddits in subreddit_topics.csv
subreddits = pd.read_csv(data_local + "subreddit_topics.csv")
bd_subreddits = subreddits[subreddits["fourth level"] == "bipolar"]
posts_ignore_case = posts[posts.subreddit_name.str.lower().isin(bd_subreddits.subreddit.str.lower())]
posts_with_case = posts[posts.subreddit_name.isin(bd_subreddits.subreddit)]
print(len(posts_ignore_case), len(posts_with_case))

# but it does matter for the subreddits from bipolar-subreddits.txt
bd_subreddits_old = pd.read_csv(data_local + "bipolar-subreddits.txt", header=None, names=["subreddit"]).squeeze(axis=0)
print(len(bd_subreddits_old))
print(bd_subreddits)
posts_bd_old = posts[posts.subreddit_name.isin(bd_subreddits_old.subreddit)]
posts_bd_old_ignore_case = posts[posts.subreddit_name.str.lower().isin(bd_subreddits_old.subreddit.str.lower())]
print(len(posts_bd_old), len(posts_bd_old_ignore_case))

# reason: three subreddits have different upper/lowercasing in bipolar-subreddits.txt
print(bd_subreddits[~bd_subreddits.subreddit.isin(bd_subreddits_old.subreddit)].subreddit)
print(bd_subreddits_old[~bd_subreddits_old.subreddit.isin(bd_subreddits.subreddit)].subreddit)

In [ ]:
# create list of unique PR terms

# re-combine spelling / grammatical variants of PR terms for analysing absences

# if any variant of a term is in the "Present" list, treat it as present, otherwise absent
# baseline variant: "-", "you, your, yourself"

PR_terms = pd.read_excel(results_folder + "Keyword-analysis/PR_terms_is_key_lemma.xlsx",
                         sheet_name="all (no pron vars), automatic").term.to_list()
print(len(PR_terms))

# get groups of variants of a head term (variants include the head term itself)
head_terms = pd.read_excel(results_folder + "Keyword-analysis/PR_terms_is_key_lemma.xlsx",
                         sheet_name="terms with variants")
head_terms["variants"] = head_terms.values.tolist()
# remove Nan values for terms that have fewer than 2, 3, 4 variants
head_terms["variants"] = head_terms["variants"].apply(lambda x: [var for var in x if pd.notna(var)])

PR_terms = [term for term in PR_terms if term not in head_terms.variant1.to_list() + head_terms.variant2.to_list() +
           head_terms.variant3.to_list() + head_terms.variant4.to_list()]

print("PR terms without variants: %d" %len(PR_terms))

### absent = does not appear in the corpus or is underused in the corpus compared to the comparison
absent = pd.read_excel(results_folder + "Keyword-analysis/PR_terms_is_key_lemma.xlsx", sheet_name="f_PR-BD == 0")
underused = pd.read_excel(results_folder + "Keyword-analysis/PR_terms_is_key_lemma.xlsx", sheet_name="underused")
absent = pd.concat([absent, underused])
print("Absent PR terms: %d" %len(absent))
# print(absent)

def reduce_pronoun_variants(term):
    # print(term)
    for placeholder, pronouns in expansions.items():
        for pronoun in pronouns:
            pattern = r"\b%s\b" % pronoun
            if re.search(pattern, term):
                term = re.sub(pattern, placeholder, term)
                return term
    return term

### present = all terms that are not absent
present = pd.read_excel(results_folder + "Keyword-analysis/PR_terms_is_key_lemma.xlsx", sheet_name="all, automatic")
print("PR terms: %d" %len(present))

present = present[~present.term.isin(absent.term)]
print("Present PR terms: %d" %len(present))

### checking pronoun variants ###
expansions = {"# you #": ["I", "you", "he", "she", "they", "we"],
              "# yourself #": ["myself", "yourself", "herself", "himself", "themself", "ourselves", "themselves"],
              "# your #": ["my", "your", "her", "his", "our", "their"]}

# replace pronoun variants with base placeholder (e.g. # your #)
absent["term"] = absent.term.apply(reduce_pronoun_variants)
absent.drop_duplicates(subset=["term"], keep="first", inplace=True)
print("Absent PR terms after dropping pronoun variants: %d" %len(absent))

present["term"] = present.term.apply(reduce_pronoun_variants)
present.drop_duplicates(subset=["term"], keep="first", inplace=True)
print("PR terms after dropping pronoun variants: %d" %len(present))

absent["pronoun_variant_is_present"] = absent.term.apply(lambda x: x in present.term.values)

### checking spelling variants ###
def check_presence_variance_group(variants):
    """
    if one of the terms of the variant group is present, mark the whole group as present
    """
    for var in variants:
        # print(var)
        if var in present.term.values:
            return True
    return False
# check what variant groups are considered present because one of their members is in present group
head_terms["present"] = head_terms.variants.apply(check_presence_variance_group)
# print(head_terms)

variants = dict(zip(head_terms.term, head_terms.variants))
# print(variants)
def check_for_head_term(term):
    for head_term, curr_variants in variants.items():
        if term in curr_variants:
            return head_term
            #print("Found term with variants: %s: %s" %(term, str(head_terms[head_terms.term == head_term].present.iloc[0])))
            #return head_terms[head_terms.term == head_term].present.iloc[0]
    return term


In [ ]:
# calculate keyness + select key lemmas